In [1]:
import pandas as pd
from datetime import datetime
from io import BytesIO
from google.cloud import storage, bigquery
import numpy as np
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

### Reporte Emision Acsel X

In [2]:
df_rep_emision_acselX= pd.read_csv('C:/data/Reportes de Emision/PACTIVAS_63886903_01_9013.TXT', sep="®", dtype=str, encoding="latin1", engine="python")
df_rep_emision_acselX.columns = df_rep_emision_acselX.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True)

In [ ]:
df_rep_emision_acselX.rename(columns={
    'FECMOV': 'FECHA_EMISION', 'TIPOOP': 'TIPO_MOVIMIENTO', 'CODPROD': 'CODIGO_PRODUCTO', 
    'FECINI': 'FECHA_INICIO', 'FECFIN': 'FECHA_FIN', 'CODPOL': 'CODIGO_POLIZA', 'NUMPOL': 
    'NRO_POLIZA', 'NUMCERT': 'NRO_CERT', 'STSCERT': 'ESTADO_DOC', 'CODMONEDA': 'MONEDA',  
    'MTOOPER': 'PRIMA_BRUTA', 'NUMOPER': 'NRO_OPERACION', 'NUMDOC': 'NRO_DOC_EMISION'
}, inplace=True)

In [4]:
df_rep_emision_acselX= df_rep_emision_acselX[['CODIGO_PRODUCTO','TIPO_MOVIMIENTO', 'FECHA_INICIO', 'FECHA_FIN', 'MONEDA', 
                                              'PRIMA_BRUTA','CODIGO_POLIZA', 'NRO_POLIZA','NRO_CERT','NRO_OPERACION', 
                                              'NRO_DOC_EMISION', 'FECHA_EMISION','ESTADO_DOC']]

In [5]:
df_rep_emision_acselX['MONEDA'].value_counts()

MONEDA
SOL    149
Name: count, dtype: int64

In [6]:
df_rep_emision_acselX['FECHA_INICIO'] = pd.to_datetime(df_rep_emision_acselX['FECHA_INICIO'], format="%d/%m/%Y", errors='coerce').dt.date
df_rep_emision_acselX['FECHA_FIN'] = pd.to_datetime(df_rep_emision_acselX['FECHA_FIN'], format="%d/%m/%Y", errors='coerce').dt.date
df_rep_emision_acselX['FECHA_EMISION'] = pd.to_datetime(df_rep_emision_acselX['FECHA_EMISION'], format="%d/%m/%Y", errors='coerce').dt.date
df_rep_emision_acselX['PRIMA_BRUTA'] = pd.to_numeric(df_rep_emision_acselX['PRIMA_BRUTA'], errors="coerce").astype('float64')

In [7]:
df_rep_emision_acselX.head(3)

,CODIGO_PRODUCTO,TIPO_MOVIMIENTO,FECHA_INICIO,FECHA_FIN,MONEDA,PRIMA_BRUTA,CODIGO_POLIZA,NRO_POLIZA,NRO_CERT,NRO_OPERACION,NRO_DOC_EMISION,FECHA_EMISION,ESTADO_DOC
0,9013,MOD,2009-05-01,2028-08-19,SOL,8.08,9013,500000,2160,4743288578,1176821456,2026-03-25,ACT
1,9013,MOD,2009-05-01,2028-08-19,SOL,8.08,9013,500000,2180,4743288578,1176821457,2026-03-25,ACT
2,9013,MOD,2009-05-01,2028-08-19,SOL,8.08,9013,500000,2335,4743288578,1176821467,2026-03-25,ACT


In [15]:
df_rep_emision_acselX.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149 entries, 0 to 148
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CODIGO_PRODUCTO  149 non-null    object 
 1   TIPO_MOVIMIENTO  149 non-null    object 
 2   FECHA_INICIO     149 non-null    object 
 3   FECHA_FIN        148 non-null    object 
 4   MONEDA           149 non-null    object 
 5   PRIMA_BRUTA      149 non-null    float64
 6   CODIGO_POLIZA    149 non-null    object 
 7   NRO_POLIZA       149 non-null    object 
 8   NRO_CERT         149 non-null    object 
 9   NRO_OPERACION    149 non-null    object 
 10  NRO_DOC_EMISION  149 non-null    object 
 11  FECHA_EMISION    149 non-null    object 
 12  ESTADO_DOC       149 non-null    object 
dtypes: float64(1), object(12)
memory usage: 15.3+ KB


### Reporte Emision SAS

In [17]:
df_reporte_emision_SAS= pd.read_csv('c:/data/Reportes de Emision/VIDA BBVA MARZO 2503_97993.csv', sep=',', encoding='latin-1', dtype=str, skiprows=7)
df_reporte_emision_SAS.columns = df_reporte_emision_SAS.columns.str.strip().str.upper().str.replace(r'[^A-Za-z0-9]', '_', regex=True)

In [18]:
df_reporte_emision_SAS.rename(columns={
    'C_DIGO_PRODUCTO':'CODIGO_PRODUCTO','C_D__CERTIFICADO_CANAL': 'CERTIFICADO_CANAL',  
    'FEC__INICIO': 'FECHA_INICIO', 'FEC__FIN': 'FECHA_FIN', 'LINEA_TRAMA_TEXTO_COMPLETO_': 'LINEA_TRAMA',
    'DESCRIPCI_N_DE_ESTADO': 'DESCRIPCION_ESTADO', 'NUMERO_DE_POLIZA': 'NRO_POLIZA', 
    'TIPO_DOC_EMISI_N': 'TIPO_DOC_EMISION', 'N__DOC_EMISI_N': 'NRO_DOC_EMISION','FECHA_DE_EMISI_N': 'FECHA_EMISION', 
    'ESTADO_DE_DOCUMENTO': 'ESTADO_DOC'
}, inplace=True)

In [19]:
df_reporte_emision_SAS= df_reporte_emision_SAS[['CODIGO_PRODUCTO','CERTIFICADO_CANAL','FECHA_INICIO','FECHA_FIN',
                                            'TIPO_MOVIMIENTO','MONEDA','PRIMA_BRUTA','DESCRIPCION_ESTADO','NRO_POLIZA',
                                            'NRO_DOC_EMISION', 'TIPO_DOC_EMISION','FECHA_EMISION','ESTADO_DOC',
                                            'LINEA_TRAMA']]

In [20]:
df_reporte_emision_SAS['CERTIFICADO_CANAL']= df_reporte_emision_SAS['CERTIFICADO_CANAL'].str[1:]
df_reporte_emision_SAS['NRO_POLIZA']= df_reporte_emision_SAS['NRO_POLIZA'].str[1:]
df_reporte_emision_SAS['TIPO_MOVIMIENTO']= df_reporte_emision_SAS['TIPO_MOVIMIENTO'].str.upper()
df_reporte_emision_SAS['DESCRIPCION_ESTADO']= df_reporte_emision_SAS['DESCRIPCION_ESTADO'].str.upper()
df_reporte_emision_SAS['FECHA_INICIO'] = pd.to_datetime(df_reporte_emision_SAS['FECHA_INICIO'], format="%d/%m/%Y", errors='coerce').dt.date
df_reporte_emision_SAS['FECHA_FIN'] = pd.to_datetime(df_reporte_emision_SAS['FECHA_FIN'], format="%d/%m/%Y", errors='coerce').dt.date
df_reporte_emision_SAS['FECHA_EMISION'] = pd.to_datetime(df_reporte_emision_SAS['FECHA_EMISION'], format='%d-%b-%y', errors='coerce').dt.date
df_reporte_emision_SAS['PRIMA_BRUTA'] = pd.to_numeric(df_reporte_emision_SAS['PRIMA_BRUTA'], errors="coerce").astype('float64')

In [21]:
df_reporte_emision_SAS['MONEDA'].value_counts()

MONEDA
USD    1411
Name: count, dtype: int64

In [22]:
df_reporte_emision_SAS.head(3)

,CODIGO_PRODUCTO,CERTIFICADO_CANAL,FECHA_INICIO,FECHA_FIN,TIPO_MOVIMIENTO,MONEDA,PRIMA_BRUTA,DESCRIPCION_ESTADO,NRO_POLIZA,NRO_DOC_EMISION,TIPO_DOC_EMISION,FECHA_EMISION,ESTADO_DOC,LINEA_TRAMA
0,3581,00110223554000814715,2026-03-17,2026-04-17,RENOVACION,USD,22.0,EMITIDO EN A/X,0101837335,1784635693,LV,2026-03-19,COB,800001102235540008147150220 01U...
1,3581,00110192254000309924,2026-03-16,2026-04-16,RENOVACION,USD,22.0,EMITIDO EN A/X,0100780057,1784611181,LV,2026-03-19,COB,800001101922540003099240166 01U...
2,3581,00110203724000591636,2026-03-16,2026-04-16,RENOVACION,USD,12.0,EMITIDO EN A/X,0115798488,1784611095,LV,2026-03-19,COB,800001102037240005916360203 01U...


In [23]:
df_reporte_emision_SAS.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1411 entries, 0 to 1410
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CODIGO_PRODUCTO     1411 non-null   object 
 1   CERTIFICADO_CANAL   1411 non-null   object 
 2   FECHA_INICIO        1411 non-null   object 
 3   FECHA_FIN           1411 non-null   object 
 4   TIPO_MOVIMIENTO     1411 non-null   object 
 5   MONEDA              1411 non-null   object 
 6   PRIMA_BRUTA         1409 non-null   float64
 7   DESCRIPCION_ESTADO  1411 non-null   object 
 8   NRO_POLIZA          1411 non-null   object 
 9   NRO_DOC_EMISION     1411 non-null   object 
 10  TIPO_DOC_EMISION    1411 non-null   object 
 11  FECHA_EMISION       1409 non-null   object 
 12  ESTADO_DOC          1411 non-null   object 
 13  LINEA_TRAMA         1411 non-null   object 
dtypes: float64(1), object(13)
memory usage: 154.5+ KB


In [ ]:
schema_Rep_Emision = [
        bigquery.SchemaField("CODIGO_PRODUCTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CERTIFICADO_CANAL", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_INICIO", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("FECHA_FIN", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("TIPO_MOVIMIENTO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("MONEDA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("PRIMA_BRUTA", bigquery.enums.SqlTypeNames.NUMERIC),
        bigquery.SchemaField("DESCRIPCION_ESTADO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("CODIGO_POLIZA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_POLIZA", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_CERT", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_OPERACION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NRO_DOC_EMISION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("TIPO_DOC_EMISION", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("FECHA_EMISION", bigquery.enums.SqlTypeNames.DATE),
        bigquery.SchemaField("ESTADO_DOC", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("NOMBRE_ARCHIVO", bigquery.enums.SqlTypeNames.STRING),
        bigquery.SchemaField("LINEA_TRAMA", bigquery.enums.SqlTypeNames.STRING)     
]